# TP 1 — Mettre le cluster en service et lire HDFS### Module 1 — Fondations · Big Data M2 / Ingénieur**Durée :** 2 heures · **Travail :** individuel ou en binôme · **Noté sur 20**---## Ce que vous devez savoir faire à la fin de ce TP1. Démarrer un cluster Hadoop et **vérifier** qu'il est réellement opérationnel — pas seulement   que les conteneurs sont lancés.2. Manipuler HDFS en ligne de commande : créer, déposer, lister, inspecter, supprimer.3. **Observer directement** le découpage en blocs et la réplication, et rattacher ce que   vous voyez au cours (séance 2, sections « partitionner » et « répliquer »).4. Lire les interfaces NameNode et ResourceManager et y trouver une information précise.5. Lire un fichier HDFS depuis PySpark.6. **Provoquer une panne** et observer la réaction du système.## LivrableCe notebook complété, exporté en **HTML** (`Fichier → Enregistrer et exporter → HTML`),déposé sur l'espace de cours avant la séance suivante. Les captures d'écran demandéesdoivent être **incluses dans le notebook** (glisser-déposer dans une cellule Markdown).## Barème| Exercice | Sujet | Points ||---|---|---|| 1 | Vérification de l'environnement | 2 || 2 | Commandes HDFS | 4 || 3 | Dépôt du jeu de données | 3 || 4 | Blocs et réplication | 5 || 5 | Interfaces web | 2 || 6 | Lecture depuis PySpark | 2 || 7 | Panne d'un DataNode | 2 |> Les questions marquées **Q** attendent une réponse rédigée dans la cellule Markdown qui suit.> Une réponse sans justification vaut zéro point.## Avant de commencerLe cluster doit tourner. Depuis un terminal **sur votre machine** (pas dans ce notebook) :```bashcd BigData-2026/99-Infra/dockerdocker compose up -d```Si vous n'avez pas encore construit l'environnement, suivez `99-Infra/INSTALLATION.md`.

---# Exercice 1 — Vérifier l'environnement  *(2 points)***Objectif.** Ne jamais commencer un TP sur une infrastructure dont on n'a pas vérifié l'état.Un cluster « démarré » n'est pas forcément un cluster « fonctionnel ».**Ce que vous devez produire.** Les sorties des trois cellules ci-dessous, et la réponse à Q1.

In [3]:
# 1.1 — Versions des outils installés dans ce conteneur!java -version!hdfs version | head -2!echo "--- PySpark ---"import pysparkprint("PySpark", pyspark.__version__)
!java -version

openjdk version "17.0.20.1" 2026-08-18
OpenJDK Runtime Environment (build 17.0.20.1+1-1-deb12u1-Debian)
OpenJDK 64-Bit Server VM (build 17.0.20.1+1-1-deb12u1-Debian, mixed mode, sharing)


In [2]:
!hdfs version

Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c
Compiled by ubuntu on 2023-06-18T08:22Z
Compiled on platform linux-x86_64
Compiled with protoc 3.7.1
From source with checksum 5652179ad55f76cb287d9c633bb53bbd
This command was run using /opt/hadoop/share/hadoop/common/hadoop-common-3.3.6.jar


In [ ]:
# 1.2 — Le NameNode répond-il ? Combien de DataNodes sont vivants ?!hdfs dfsadmin -report | head -30

### Q1 *(2 pts)* — Lisez attentivement la sortie de la cellule 1.2 et répondez :- **a.** Quelle est la capacité totale (`Configured Capacity`) du cluster, en Gio ?- **b.** Combien de DataNodes sont signalés comme vivants (`Live datanodes`) ?- **c.** `DFS Used` est proche de zéro alors que `Non DFS Used` est important. Que représente  cette seconde valeur, et pourquoi est-elle si élevée ici ?*Indice pour (c) : les DataNodes partagent le disque de votre machine avec le reste du système.*

**Votre réponse :***(rédigez ici)*

---# Exercice 2 — Commandes HDFS  *(4 points)***Objectif.** Acquérir les réflexes de manipulation de HDFS. La syntaxe est volontairementproche de celle du shell Unix, mais **HDFS n'est pas un système de fichiers Unix** : c'estun service réseau, et certaines opérations qui vous semblent naturelles y sont impossibles.**Ce que vous devez produire.** Les commandes complétées, et la réponse à Q2.

In [ ]:
# 2.1 — Explorer la racine de HDFS!hdfs dfs -ls /

In [ ]:
# 2.2 — À VOUS : créez votre répertoire de travail /user/<votre_nom>#          puis un sous-répertoire brut/ à l'intérieur.#          Remplacez etudiant par votre nom de famille en minuscules.UTILISATEUR = "etudiant"   # <-- À MODIFIER# Complétez la commande (indice : hdfs dfs -mkdir, option -p)!hdfs dfs -mkdir ...

In [ ]:
# 2.3 — Déposez un petit fichier de test et relisez-le!echo "Premier fichier depose sur HDFS." > /tmp/test.txt!hdfs dfs -put -f /tmp/test.txt /user/{UTILISATEUR}/brut/!hdfs dfs -cat /user/{UTILISATEUR}/brut/test.txt

In [ ]:
# 2.4 — À VOUS : trois commandes à trouver dans la documentation#   a) afficher la taille du répertoire brut/ dans un format lisible#   b) afficher les statistiques du fichier test.txt (taille de bloc, réplication)#   c) modifier le facteur de réplication de test.txt pour le porter à 1# a)!hdfs dfs ...# b)!hdfs dfs ...# c)!hdfs dfs ...

In [ ]:
# 2.5 — Une opération qui ÉCHOUE. Exécutez, lisez le message, ne le corrigez pas.!echo "ligne ajoutee" > /tmp/suite.txt!hdfs dfs -appendToFile /tmp/suite.txt /user/{UTILISATEUR}/brut/test.txt!echo "--- contenu apres append ---"!hdfs dfs -cat /user/{UTILISATEUR}/brut/test.txt!echo "--- maintenant, une modification AU MILIEU du fichier ---"!hdfs dfs -truncate 10 /user/{UTILISATEUR}/brut/test.txt

### Q2 *(2 pts)* — HDFS permet l'ajout en fin de fichier (`appendToFile`) mais **ne permet pas**de modifier un octet au milieu d'un fichier existant.- **a.** Rattachez cette limitation à l'une des quatre hypothèses de conception de GFS  (cours §6.1 du polycopié, et article de lecture obligatoire).- **b.** Quelle conséquence cela a-t-il pour un cas d'usage bancaire où l'on voudrait  « corriger le montant d'une transaction déjà écrite » ? Comment fait-on en pratique ?

**Votre réponse :***(rédigez ici)*

---# Exercice 3 — Déposer le jeu de données de votre filière  *(3 points)***Objectif.** Produire un fichier assez volumineux pour être découpé en plusieurs blocs,condition nécessaire à l'exercice 4.Choisissez votre filière ci-dessous. Le générateur est **déterministe** : tous les étudiantsd'une même filière obtiendront exactement le même fichier, donc les mêmes chiffres.

In [ ]:
# 3.1 — Génération locale du jeu de données (2 à 4 minutes)FILIERE = "if"      # "if" = Ingénierie Financière · "an" = Art Numérique!python /home/tinku/cours/99-Infra/scripts/generate_datasets.py \        --filiere {FILIERE} --sortie /home/tinku/work/data --evenements 2000000!ls -lh /home/tinku/work/data/

In [ ]:
# 3.2 — À VOUS : déposez le fichier d'événements sur HDFS dans /user/<vous>/brut/#          puis vérifiez sa présence et sa taille.FICHIER = "if_transactions.jsonl" if FILIERE == "if" else "an_evenements.jsonl"!hdfs dfs ...!hdfs dfs ...

### Q3 *(1 pt)* — Comparez le temps de dépôt à la taille du fichier et déduisez-en un débitapproximatif en Mio/s. Ce débit vous paraît-il cohérent avec le fait que chaque bloc estécrit sur **deux** DataNodes ? Justifiez en une ou deux phrases.*Astuce : préfixez la commande par `%%time` dans une cellule dédiée pour la chronométrer.*

**Votre réponse :***(rédigez ici)*

---# Exercice 4 — Blocs et réplication  *(5 points — cœur du TP)***Objectif.** Voir de vos propres yeux ce que le cours a décrit de façon abstraite :un fichier découpé en blocs, chaque bloc répliqué sur des machines distinctes.`hdfs fsck` est l'outil d'inspection du système de fichiers. Lisez sa sortie attentivement :c'est le document le plus instructif de ce TP.

In [ ]:
# 4.1 — Vue d'ensemble : santé du fichier!hdfs fsck /user/{UTILISATEUR}/brut/{FICHIER}

In [ ]:
# 4.2 — Le détail : quels blocs, sur quelles machines ?!hdfs fsck /user/{UTILISATEUR}/brut/{FICHIER} -files -blocks -locations

### Q4 *(3 pts)* — À partir de la sortie de 4.2 :- **a.** Combien de blocs composent votre fichier ? **Retrouvez ce nombre par le calcul**  à partir de la taille du fichier et de la taille de bloc configurée (32 Mio dans ce cluster).- **b.** Quelle est la taille du **dernier** bloc ? Est-elle égale à 32 Mio ? Que vous apprend  cette observation sur ce qu'est réellement un « bloc » HDFS ?- **c.** Prenez le premier bloc de la liste. Sur quels DataNodes ses réplicas se trouvent-ils ?  Prenez maintenant le deuxième bloc : est-ce la même paire de machines ? Que peut-on en  déduire sur la stratégie de placement ?

**Votre réponse :***(rédigez ici)*

In [ ]:
# 4.3 — Comparaison : un petit fichier!hdfs dfs -put -f /tmp/test.txt /user/{UTILISATEUR}/brut/petit.txt!hdfs fsck /user/{UTILISATEUR}/brut/petit.txt -files -blocks!echo "=== Espace reellement occupe par les deux fichiers ==="!hdfs dfs -du -h /user/{UTILISATEUR}/brut/

### Q5 *(2 pts)* — Le fichier `petit.txt` fait quelques dizaines d'octets.- **a.** Combien de blocs occupe-t-il ? Combien d'espace disque consomme-t-il réellement ?- **b.** Le NameNode conserve en **mémoire vive** les métadonnées de chaque fichier, répertoire  et bloc — comptez environ **150 octets par objet**. Calculez la mémoire nécessaire au  NameNode pour stocker **10 millions de petits fichiers** d'un bloc chacun.- **c.** Comparez au cas où les mêmes données seraient regroupées en **10 000 gros fichiers**  de 1 000 blocs. Quel est le rapport ? Comment nomme-t-on ce problème, et quelle est  la parade en pratique ?

**Votre réponse :***(rédigez ici)*

---# Exercice 5 — Les interfaces web  *(2 points)***Objectif.** Savoir où chercher une information dans les interfaces d'administration.Vous y reviendrez à chaque TP.Ouvrez dans votre navigateur :| Interface | Adresse ||---|---|| NameNode HDFS | http://localhost:9870 || YARN ResourceManager | http://localhost:8088 |### Travail demandé1. Sur le NameNode, onglet **Utilities → Browse the file system**, naviguez jusqu'à votre   fichier et cliquez dessus. **Capture d'écran n° 1** : la fenêtre montrant la liste des blocs   et leur localisation.2. Sur le NameNode, onglet **Datanodes**. **Capture d'écran n° 2** : le tableau des DataNodes.3. Sur YARN, page d'accueil : relevez le nombre de nœuds actifs, la mémoire totale et la   mémoire disponible.**Insérez vos captures ci-dessous** (glisser-déposer dans la cellule Markdown).

**Capture 1 — blocs du fichier :***(déposez l'image ici)***Capture 2 — DataNodes :***(déposez l'image ici)*### Q6 *(2 pts)* — Sur la page YARN : combien de mémoire totale le cluster offre-t-il auxapplications, et pourquoi cette valeur ne correspond-elle pas à la RAM de votre machine ?

**Votre réponse :***(rédigez ici)*

---# Exercice 6 — Premier contact avec PySpark  *(2 points)***Objectif.** Lire un fichier HDFS depuis Spark. Nous n'expliquons pas encore *comment*Spark fonctionne — c'est le module 3. L'objectif ici est seulement de vérifier que la chaîneSpark → HDFS est opérationnelle, et d'observer un premier chiffre.

In [ ]:
# 6.1 — Créer une session Sparkfrom pyspark.sql import SparkSessionspark = (SparkSession.builder         .appName("TP1 - Premier contact")         .master("local[*]")                     # exécution locale : YARN viendra au module 3         .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")         .getOrCreate())spark.sparkContext.setLogLevel("WARN")print("Spark", spark.version)

In [ ]:
# 6.2 — À VOUS : lire le fichier JSON depuis HDFS et afficher#          le nombre de lignes, puis les 5 premières.chemin = f"hdfs://namenode:8020/user/{UTILISATEUR}/brut/{FICHIER}"df = spark.read...print(...)df.show(5)

In [ ]:
# 6.3 — Combien de partitions Spark a-t-il créées ?print("Partitions :", df.rdd.getNumPartitions())

### Q7 *(2 pts)* — Comparez le nombre de partitions Spark au nombre de blocs HDFS trouvéen Q4. Que constatez-vous, et comment l'expliquez-vous ?

**Votre réponse :***(rédigez ici)*

---# Exercice 7 — Provoquer une panne  *(2 points)***Objectif.** Vérifier expérimentalement l'affirmation du cours : « sur un cluster, la panneest un régime normal que le système absorbe sans intervention ».**Attention :** exécutez les commandes `docker` depuis un **terminal de votre machine**,pas depuis ce notebook — le conteneur Jupyter n'a pas accès au démon Docker.

### Marche à suivre1. Notez l'état actuel :   ```   hdfs fsck /user/<vous>/brut/<fichier>   ```   (exécutez la cellule 7.1 ci-dessous)2. Dans un terminal **de votre machine** :   ```bash   docker compose stop datanode2   ```3. Attendez **environ 30 secondes**, puis exécutez la cellule 7.2.4. Rallumez le nœud :   ```bash   docker compose start datanode2   ```5. Attendez à nouveau, puis exécutez la cellule 7.3.

In [ ]:
# 7.1 — État initial!hdfs dfsadmin -report | grep -E "Live datanodes|Dead datanodes|Under replicated"!hdfs fsck /user/{UTILISATEUR}/brut/{FICHIER} | tail -20

In [ ]:
# 7.2 — Après l'arrêt de datanode2 (attendre ~30 s)!hdfs dfsadmin -report | grep -E "Live datanodes|Dead datanodes"!hdfs fsck /user/{UTILISATEUR}/brut/{FICHIER} | tail -20print("--- Le fichier est-il toujours lisible ? ---")!hdfs dfs -cat /user/{UTILISATEUR}/brut/petit.txt

In [ ]:
# 7.3 — Après le redémarrage de datanode2!hdfs dfsadmin -report | grep -E "Live datanodes|Dead datanodes"!hdfs fsck /user/{UTILISATEUR}/brut/{FICHIER} | tail -20

### Q8 *(2 pts)* —- **a.** Pendant l'arrêt de `datanode2`, le fichier était-il toujours lisible ? Pourquoi ?- **b.** Quel statut `fsck` attribuait-il aux blocs pendant la panne ? Le fichier était-il  signalé comme corrompu (`CORRUPT`) ou comme sous-répliqué (`Under-replicated`) ?  Quelle différence essentielle y a-t-il entre ces deux états ?- **c.** Le NameNode déclare un DataNode mort après **10 minutes 30** de silence par défaut,  pas après 30 secondes. Pourquoi un délai aussi long ? Que se passerait-il si ce délai  était de 5 secondes ?

**Votre réponse :***(rédigez ici)*

---# SynthèseRécapitulez en quelques lignes ce que ce TP vous a permis d'**observer directement**, en lereliant aux concepts de la séance 2. Pour chaque mécanisme du cours, indiquez à quel momentdu TP vous l'avez vu à l'œuvre.| Concept du cours | Où l'avez-vous observé ? ||---|---|| Partitionnement en blocs | || Réplication | || Détection de panne | || Tolérance aux pannes | || Localité des données | || Problème des petits fichiers | |

In [ ]:
# Nettoyage (optionnel) — libère l'espace HDFS et arrête la session Spark# !hdfs dfs -rm -r -skipTrash /user/{UTILISATEUR}spark.stop()print("Session Spark fermee.")

---## Avant de rendre- [ ] Toutes les cellules ont été exécutées **dans l'ordre**, et leurs sorties sont visibles.- [ ] Les huit questions **Q1 à Q8** ont une réponse rédigée et justifiée.- [ ] Les **deux captures d'écran** sont insérées dans le notebook.- [ ] Le tableau de synthèse est complété.- [ ] Le notebook est exporté en **HTML** et déposé sur l'espace de cours.**Bon TP.**